In [1]:
import polars as pl

# Input: the reduced loan-level file from the research stage
df = pl.read_parquet("../data/processed/fannie_2017_loan_level.parquet")

# Confirm we loaded what we expect
print(f"Rows: {df.height:,}")
print(f"Features: {df.width}")

# Output path for the engineered feature set (kept distinct from the source)
OUTPUT_PATH = "../data/processed/fannie_2017_features.parquet"

Rows: 2,046,851
Features: 115


In [2]:
# Check which termination and timing fields are still present and populated
timing_cols = [
    "LOAN_AGE",
    "ZB_DTE",           # zero balance (payoff/termination) date
    "ORIG_DATE",        # origination date
    "REM_MONTHS",       # remaining months to maturity
    "MATR_DT",          # maturity date
    "Zero_Bal_Code",    # why the loan terminated
]

present = [c for c in timing_cols if c in df.columns]
print("Timing fields present:", present)

# For the ones present, see how populated they are
for c in present:
    non_null = df.filter(pl.col(c).is_not_null()).height
    pct = non_null / df.height * 100
    print(f"{c:<16} {non_null:>10,} populated ({pct:.1f}%)")

Timing fields present: ['LOAN_AGE', 'ZB_DTE', 'ORIG_DATE', 'REM_MONTHS', 'MATR_DT']
LOAN_AGE          2,046,851 populated (100.0%)
ZB_DTE                    0 populated (0.0%)
ORIG_DATE         2,046,851 populated (100.0%)
REM_MONTHS        2,046,851 populated (100.0%)
MATR_DT           2,046,851 populated (100.0%)


## Interest Income Per Loan (Amortized, 7-Year Horizon)

Estimates the interest income each loan generates, to feed the expected-return
calculation in the LP stage.

Assumptions and rationale:
- **Amortized interest.** We compute the fixed monthly payment from the loan's
  rate, original balance, and full 360-month term, then sum the interest portion
  of the payments. Mortgage interest is front-loaded, so amortization captures
  the real income shape better than a flat approximation.
- **7-year effective horizon (84 payments).** We sum interest only through month
  84, not the full 30 years. Most 30-year mortgages terminate early through sale
  or refinance, so ~7 years reflects realized loan life. This also serves as a
  proxy for prepayment, which we otherwise ignore.
- **Payment set on full term.** The monthly payment uses the true 360-month
  schedule, since that is the borrower's actual contractual payment. We simply
  truncate the interest sum at year 7.
- **No discounting.** Interest is summed in nominal dollars, not present value.
  A simplification for this build; discounting is noted as possible future work.
- **Gross interest.** Servicing and

In [4]:
# Cast the inputs we need to numeric
df = df.with_columns([
    pl.col("ORIG_RATE").cast(pl.Float64, strict=False),
    pl.col("ORIG_UPB").cast(pl.Float64, strict=False),
    pl.col("ORIG_TERM").cast(pl.Float64, strict=False),
])

FULL_TERM = 360      # months, used to set the contractual payment
HORIZON = 84         # months, 7-year effective holding period

# Monthly interest rate from the annual rate (percent -> decimal -> monthly)
# Standard amortization: M = P * r * (1+r)^n / ((1+r)^n - 1)
df = df.with_columns(
    (pl.col("ORIG_RATE") / 100 / 12).alias("monthly_rate")
)

df = df.with_columns(
    (
        pl.col("ORIG_UPB")
        * pl.col("monthly_rate")
        * (1 + pl.col("monthly_rate")).pow(FULL_TERM)
        / ((1 + pl.col("monthly_rate")).pow(FULL_TERM) - 1)
    ).alias("monthly_payment")
)

### Vectorized Cumulative Interest (Closed-Form)

Computes interest income over the 84-month horizon in closed form, no per-loan
loop. Logic: total paid over 84 months is (monthly payment x 84). The principal
retired over those months has a closed-form expression, so interest is the
difference between total paid and principal retired.

In [5]:
# Balance remaining after HORIZON payments (closed-form amortization):
#   B_n = P * ( (1+r)^N - (1+r)^n ) / ( (1+r)^N - 1 )
# Principal retired over the horizon = ORIG_UPB - B_n
# Total paid over horizon = monthly_payment * HORIZON
# Interest income = total paid - principal retired

df = df.with_columns(
    (
        pl.col("ORIG_UPB")
        * ((1 + pl.col("monthly_rate")).pow(FULL_TERM) - (1 + pl.col("monthly_rate")).pow(HORIZON))
        / ((1 + pl.col("monthly_rate")).pow(FULL_TERM) - 1)
    ).alias("balance_after_horizon")
)

df = df.with_columns([
    (pl.col("ORIG_UPB") - pl.col("balance_after_horizon")).alias("principal_retired"),
    (pl.col("monthly_payment") * HORIZON).alias("total_paid_horizon"),
])

df = df.with_columns(
    (pl.col("total_paid_horizon") - pl.col("principal_retired")).alias("interest_income_7yr")
)

# Quick sanity check
print(df.select([
    "ORIG_UPB", "ORIG_RATE", "monthly_payment",
    "interest_income_7yr"
]).describe())

shape: (9, 5)
┌────────────┬───────────────┬────────────┬─────────────────┬─────────────────────┐
│ statistic  ┆ ORIG_UPB      ┆ ORIG_RATE  ┆ monthly_payment ┆ interest_income_7yr │
│ ---        ┆ ---           ┆ ---        ┆ ---             ┆ ---                 │
│ str        ┆ f64           ┆ f64        ┆ f64             ┆ f64                 │
╞════════════╪═══════════════╪════════════╪═════════════════╪═════════════════════╡
│ count      ┆ 2.046851e6    ┆ 2.046851e6 ┆ 2.046851e6      ┆ 2.046851e6          │
│ null_count ┆ 0.0           ┆ 0.0        ┆ 0.0             ┆ 0.0                 │
│ mean       ┆ 228839.312192 ┆ 4.140495   ┆ 1111.131769     ┆ 61984.344905        │
│ std        ┆ 119551.768064 ┆ 0.494707   ┆ 586.493053      ┆ 33637.694073        │
│ min        ┆ 5000.0        ┆ 1.79       ┆ 23.15578        ┆ 1222.987564         │
│ 25%        ┆ 137000.0      ┆ 3.875      ┆ 664.118853      ┆ 36689.626906        │
│ 50%        ┆ 206000.0      ┆ 4.125      ┆ 1001.562886     ┆ 

## LGD Flat-Value Column

Attaches a Loss Given Default (LGD) value to each loan for the expected-loss
calculation in the LP and simulation stages.

Assumptions and rationale:
- **Flat value, not computed.** The Qi-Yang formula could not be applied here,
  since all recovery fields (net sales proceeds, foreclosure costs, etc.) are
  null across the full book. So we assign a flat LGD from the literature.
- **30% baseline.** Sirignano et al. (2016) use 30% for a normal economy and
  50% for a downturn. Our 2017 vintage sits in a stable housing period, so 30%
  is the baseline. The 50% value is reserved as a downturn stress scenario for
  the simulation stage.
- **Single tunable parameter.** LGD is set once, at the top, so it can be
  changed in one place and the column rebuilt.

In [6]:
# LGD assumption. Change this one value to adjust the baseline (e.g. 0.50 for
# the downturn stress scenario).
LGD_BASELINE = 0.30

df = df.with_columns(
    pl.lit(LGD_BASELINE).alias("lgd")
)

# Quick confirmation
print(f"LGD applied: {LGD_BASELINE:.0%}")
print(df.select("lgd").head(3))

LGD applied: 30%
shape: (3, 1)
┌─────┐
│ lgd │
│ --- │
│ f64 │
╞═════╡
│ 0.3 │
│ 0.3 │
│ 0.3 │
└─────┘


## Credit Grade Buckets

Groups loans into standard FICO credit bands from CSCORE_B, for use in EDA and
as a potential diversification dimension in the LP stage.

Definitions and rationale:
- **Standard FICO bands** are used for defensibility:
  - Exceptional: 800+
  - Very Good: 740 to 799
  - Good: 670 to 739
  - Fair/Poor (combined): below 670
- **Poor and Fair are combined.** The Poor band (below 580) holds only ~1,574
  loans and behaves erratically due to its small size, so merging it with Fair
  avoids noise while keeping the low-credit segment represented.
- **Source column:** CSCORE_B, the Borrower Credit Score at Origination, which
  the Fannie Mae glossary defines as the Classic FICO score.

In [9]:
df = df.with_columns(pl.col("CSCORE_B").cast(pl.Int32, strict=False))

df = df.with_columns(
    pl.when(pl.col("CSCORE_B").is_null()).then(pl.lit("Unknown"))
    .when(pl.col("CSCORE_B") >= 800).then(pl.lit("Exceptional"))
    .when(pl.col("CSCORE_B") >= 740).then(pl.lit("Very Good"))
    .when(pl.col("CSCORE_B") >= 670).then(pl.lit("Good"))
    .otherwise(pl.lit("Fair/Poor"))
    .alias("credit_grade")
)

print(df["credit_grade"].value_counts().sort("count", descending=True))

shape: (5, 2)
┌──────────────┬────────┐
│ credit_grade ┆ count  │
│ ---          ┆ ---    │
│ str          ┆ u32    │
╞══════════════╪════════╡
│ Very Good    ┆ 967786 │
│ Good         ┆ 623257 │
│ Exceptional  ┆ 308739 │
│ Fair/Poor    ┆ 145496 │
│ Unknown      ┆ 1573   │
└──────────────┴────────┘


## Loss If Default (Dollar Loss Per Loan)

Computes the dollar loss a loan would incur if it defaults, for the
expected-return calculation in the LP stage.

Logic and scope:
- **loss_if_default = LGD x ORIG_UPB.** The fraction lost (LGD) times the loan
  balance gives the dollar loss on default.
- **PD not applied here.** Expected loss is PD x LGD x loan amount. PD comes
  from the ML stage, so we build the LGD x amount portion now and apply PD
  during post-ML assembly.
- Uses the flat LGD baseline set earlier (LGD_BASELINE).

In [10]:
df = df.with_columns(
    (pl.col("lgd") * pl.col("ORIG_UPB")).alias("loss_if_default")
)

print(df.select(["ORIG_UPB", "lgd", "loss_if_default"]).describe())

shape: (9, 4)
┌────────────┬───────────────┬────────────┬─────────────────┐
│ statistic  ┆ ORIG_UPB      ┆ lgd        ┆ loss_if_default │
│ ---        ┆ ---           ┆ ---        ┆ ---             │
│ str        ┆ f64           ┆ f64        ┆ f64             │
╞════════════╪═══════════════╪════════════╪═════════════════╡
│ count      ┆ 2.046851e6    ┆ 2.046851e6 ┆ 2.046851e6      │
│ null_count ┆ 0.0           ┆ 0.0        ┆ 0.0             │
│ mean       ┆ 228839.312192 ┆ 0.3        ┆ 68651.793658    │
│ std        ┆ 119551.768064 ┆ 2.4832e-18 ┆ 35865.530419    │
│ min        ┆ 5000.0        ┆ 0.3        ┆ 1500.0          │
│ 25%        ┆ 137000.0      ┆ 0.3        ┆ 41100.0         │
│ 50%        ┆ 206000.0      ┆ 0.3        ┆ 61800.0         │
│ 75%        ┆ 300000.0      ┆ 0.3        ┆ 90000.0         │
│ max        ┆ 1.223e6       ┆ 0.3        ┆ 366900.0        │
└────────────┴───────────────┴────────────┴─────────────────┘


## Socio-Economic Constraint Indicators

Converts the three socio-economic flags into 0/1 integer columns for direct use
in Gurobi constraint sums. Kept as separate columns to preserve the distinction
between programs, since first-time buyer, HomeReady, and HFA Preferred each
behave differently.

- is_first_time: 1 if FIRST_FLAG is Y, else 0
- is_homeready: 1 if HOMEREADY_PROGRAM_INDICATOR is H, else 0
- is_hfa: 1 if HOMEREADY_PROGRAM_INDICATOR is F, else 0

State and credit grade are left as labels, to be grouped directly in the LP
stage rather than one-hot encoded here.

In [11]:
df = df.with_columns([
    (pl.col("FIRST_FLAG") == "Y").cast(pl.Int8).alias("is_first_time"),
    (pl.col("HOMEREADY_PROGRAM_INDICATOR") == "H").cast(pl.Int8).alias("is_homeready"),
    (pl.col("HOMEREADY_PROGRAM_INDICATOR") == "F").cast(pl.Int8).alias("is_hfa"),
])

# Confirm counts match what we saw in research
print(df.select([
    pl.col("is_first_time").sum().alias("first_time_count"),
    pl.col("is_homeready").sum().alias("homeready_count"),
    pl.col("is_hfa").sum().alias("hfa_count"),
]))

shape: (1, 3)
┌──────────────────┬─────────────────┬───────────┐
│ first_time_count ┆ homeready_count ┆ hfa_count │
│ ---              ┆ ---             ┆ ---       │
│ i64              ┆ i64             ┆ i64       │
╞══════════════════╪═════════════════╪═══════════╡
│ 488486           ┆ 103860          ┆ 52205     │
└──────────────────┴─────────────────┴───────────┘


In [12]:
# Review final dataset size and confirm the new feature columns are present
print(f"Rows: {df.height:,}")
print(f"Features: {df.width}")

new_features = [
    "interest_income_7yr",
    "lgd",
    "loss_if_default",
    "credit_grade",
    "is_first_time",
    "is_homeready",
    "is_hfa",
]
print("\nNew feature columns present:")
print([c for c in new_features if c in df.columns])

Rows: 2,046,851
Features: 127

New feature columns present:
['interest_income_7yr', 'lgd', 'loss_if_default', 'credit_grade', 'is_first_time', 'is_homeready', 'is_hfa']


In [13]:
# Drop the scratch intermediates, keep monthly_payment for interpretability
scratch_cols = [
    "monthly_rate",
    "balance_after_horizon",
    "principal_retired",
    "total_paid_horizon",
]
df = df.drop([c for c in scratch_cols if c in df.columns])

print(f"Rows: {df.height:,}")
print(f"Features: {df.width}")

Rows: 2,046,851
Features: 123


## Save the Engineered Feature Set

Saves the finished dataset to its own file, so the original reduction file stays untouched.

Where the columns come from:
- Input: fannie_2017_loan_level.parquet, 117 columns (113 raw Fannie fields plus max_dlq_ever, zero_bal_code, default_flag, orig_quarter added during reduction).
- Added 8 columns here: monthly_payment, interest_income_7yr, lgd, loss_if_default, credit_grade, is_first_time, is_homeready, is_hfa.
- Dropped 4 scratch columns from the interest math: monthly_rate, balance_after_horizon, principal_retired, total_paid_horizon.
- Output: 123 columns.

Features we built and the assumptions behind them:
- interest_income_7yr: amortized interest over a 7-year (84-month) window. The monthly payment is set on the full 360-month term, but we only sum interest through month 84. This reflects that most 30-year mortgages end early through a sale or refinance, and stands in for prepayment. No discounting, gross interest only.
- lgd: flat 30 percent (Sirignano et al. 2016). The recovery fields are empty across the whole book, so the Qi-Yang formula cannot be used. Set as one tunable value (LGD_BASELINE); 50 percent is held back for a downturn stress test.
- loss_if_default: lgd times ORIG_UPB, the dollar loss if a loan defaults. PD gets applied later to turn this into expected loss.
- credit_grade: standard FICO bands (Exceptional, Very Good, Good, Fair/Poor), with Poor and Fair combined and a separate Unknown bucket for missing scores.
- is_first_time, is_homeready, is_hfa: 0/1 flags for the socio-economic constraints, kept separate so each program stays distinct.

In [15]:
OUTPUT_PATH = "../data/processed/fannie_2017_features_added.parquet"
df.write_parquet(OUTPUT_PATH)
print(f"Saved {df.height:,} rows x {df.width} columns to {OUTPUT_PATH}")

Saved 2,046,851 rows x 123 columns to ../data/processed/fannie_2017_features_added.parquet
